# Implement self-attention with trainable weights

In [3]:
import re
import torch

Let's assume we have the following input sentence:

```python
text = "Your journey starts with one step"
```
Let's assume tokens have been already identified (one word = one token):

```python
tokens = ['Your', 'journey', 'starts', 'with', 'one', 'step']
```

A vocabulary has been crated and token ids have been assigned to each token:

```python
vocab = {token: idx for idx, token in enumerate(tokens)}
```

In [ ]:
tokens = ['Your', 'journey', 'starts', 'with', 'one', 'step']
vocab = {token: idx for idx, token in enumerate(tokens)}
print(vocab)

{'Your': 0, 'journey': 1, 'starts': 2, 'with': 3, 'one': 4, 'step': 5}


Given our vocabulary:

```python 
{'Your': 0, 'journey': 1, 'starts': 2, 'with': 3, 'one': 4, 'step': 5}
```
we have performed token embeddings using 3-dimension vectors:

```

0 -->   [0.43, 0.15, 0.89]  # Your     (x^1)
1 -->   [0.55, 0.87, 0.66]  # journey  (x^2)
2 -->   [0.57, 0.85, 0.64]  # starts   (x^3)
3 -->   [0.22, 0.58, 0.33]  # with     (x^4)
4 -->   [0.77, 0.25, 0.10]  # one      (x^5)   
5 -->   [0.05, 0.80, 0.55]  # step     (x^6)


```

In [5]:
# import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.43, 0.15, 0.89], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

### Step 1 - Converting Input embeddings into Key, Query, Value vectors

In [11]:
# setting variables
x_2 = inputs[1]         #A this is the second token of our input sequence, 'journey'
d_in = inputs.shape[1]  #B this is the dimensionality of our token embeddings, which is 3 in this case (input embeddings)
d_out = 2               #C this is the dimensionality of our output context vector, which is 2 in this case (output embeddings)


#### 1.1 Initialize W<sub>q</sub>, W<sub>k</sub> and W<sub>v</sub> matrixes

In [ ]:

# initialize Wq, Wk, Wv matrixes with random values
torch.manual_seed(123)
#this is the weight matrix for the query, which will transform our input embeddings into query vectors
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
#this is the weight matrix for the key, which will transform our input embeddings into key vectors
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
#this is the weight matrix for the value, which will transform our input embeddings into value vectors
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)


# requires_grad=False because we are not training the model, we just want to see the output of the attention mechanism with random weights
# if we were training the model, we would set requires_grad=True and use an optimizer to update the weights based on the loss function

print(f"W_query:\n{W_query}\n")
print(f"W_key:\n{W_key}\n")
print(f"W_value:\n{W_value}\n")

#### 1.2 Obtain Query, Key and Values matrixes from Inputs

In [8]:
keys = inputs @ W_key
values = inputs @ W_value
queries = inputs @ W_query

print(f"Keys:\n{keys}\n")
print(f"Values:\n{values}\n")
print(f"Queries:\n{queries}\n")

Keys:
tensor([[0.5029, 0.7447],
        [0.5029, 0.7447],
        [0.5961, 1.0120],
        [0.3315, 0.6018],
        [0.2366, 0.2906],
        [0.4569, 0.8729]])

Values:
tensor([[1.0021, 1.1026],
        [1.0021, 1.1026],
        [1.0798, 1.1507],
        [0.5105, 0.5851],
        [0.8195, 0.6322],
        [0.5305, 0.7352]])

Queries:
tensor([[1.0047, 1.0054],
        [1.0047, 1.0054],
        [0.9247, 1.0431],
        [0.4587, 0.5395],
        [0.4696, 0.5213],
        [0.6009, 0.7031]])



### Step 2 - Calculate the attention scores

To calculate attention scores we perfor dot product between the query matrix (Q) and the keys matrix transposed (K.T)

In [14]:
attn_scores = queries @ keys.T
print(f"Attention scores matrix:({attn_scores.shape[0]}x{attn_scores.shape[1]})\n{attn_scores}\n")

Attention scores matrix:(6x6)
tensor([[1.2540, 1.2540, 1.6164, 0.9381, 0.5299, 1.3367],
        [1.2540, 1.2540, 1.6164, 0.9381, 0.5299, 1.3367],
        [1.2418, 1.2418, 1.6069, 0.9343, 0.5219, 1.3331],
        [0.6324, 0.6324, 0.8194, 0.4767, 0.2653, 0.6805],
        [0.6244, 0.6244, 0.8075, 0.4694, 0.2626, 0.6696],
        [0.8258, 0.8258, 1.0697, 0.6223, 0.3465, 0.8883]])



### Step 3 - Calculate the attention weights

We normalize attention scores to determine attention weights.

In [15]:
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(f"Attention weights matrix:({attn_weights.shape[0]}x{attn_weights.shape[1]})\n{attn_weights}\n")

Attention weights matrix:(6x6)
tensor([[0.1739, 0.1739, 0.2247, 0.1391, 0.1042, 0.1843],
        [0.1739, 0.1739, 0.2247, 0.1391, 0.1042, 0.1843],
        [0.1734, 0.1734, 0.2245, 0.1395, 0.1042, 0.1850],
        [0.1711, 0.1711, 0.1953, 0.1533, 0.1320, 0.1771],
        [0.1712, 0.1712, 0.1949, 0.1534, 0.1326, 0.1768],
        [0.1721, 0.1721, 0.2045, 0.1490, 0.1226, 0.1798]])



### Step 4 - Compute context vectors

In [19]:
context_vector = attn_weights @ values
print(f"Context vectors:\n{context_vector}\n")
print(f"Context vector for 'journey':\n{context_vector[1]}")

Context vectors:
tensor([[0.8452, 0.9247],
        [0.8452, 0.9247],
        [0.8447, 0.9242],
        [0.8343, 0.9055],
        [0.8342, 0.9053],
        [0.8375, 0.9116]])

Context vector for 'journey':
tensor([0.8452, 0.9247])


In [20]:
class SelfAttention(torch.nn.Module):
    def __init__(self, d_in, d_out):
        super(SelfAttention, self).__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=True)

    def forward(self, inputs):
        keys = inputs @ self.W_key
        values = inputs @ self.W_value
        queries = inputs @ self.W_query

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vector = attn_weights @ values

        return context_vector

In [21]:
torch.manual_seed(123)
self_attn = SelfAttention(d_in, d_out)
print(self_attn(inputs))

tensor([[0.2596, 0.7736],
        [0.2596, 0.7736],
        [0.2644, 0.7854],
        [0.2558, 0.7647],
        [0.2542, 0.7611],
        [0.2591, 0.7726]], grad_fn=<MmBackward0>)


In [ ]:
class SelfAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, qvk_bias=False):
        super(SelfAttention, self).__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qvk_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qvk_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qvk_bias)

    def forward(self, inputs):
        keys = inputs @ self.W_key
        values = inputs @ self.W_value
        queries = inputs @ self.W_query

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vector = attn_weights @ values

        return context_vector